# Customer Segmentation & Marketing Campaign Analytics
## Phase 5 — Dashboard Preparation

**Goal of this notebook.** Export one flat, ready-to-load file for the Tableau Public dashboard,
and confirm it carries everything the dashboard needs to show: segment sizes, response rates by
segment, and at least one filter.

**Why a separate export, not a direct database connection.** Tableau Public (the free tier used
here) cannot connect live to a local DuckDB file — it expects a flat file (CSV/Excel) or one of
its supported live connectors, neither of which DuckDB is on the free tier. So the handoff point
between the SQL/Python side of this project and the dashboard is a single CSV, built by SQL and
exported here, that Tableau reads directly. This keeps the same "SQL does the shaping, the tool on
top does the visualizing" split used throughout the project.

**What building the dashboard itself needs.** Tableau Public Desktop is a native application that
has to be downloaded, opened, and worked in directly — it cannot be driven from this notebook or
by an AI agent. `dashboard/README.md`, written alongside this notebook, is the exact field-by-field
build guide for that part.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.db import connect, run_sql_file

pd.set_option("display.width", 200)
con = connect()

## 1. Building the export

`sql/15_dashboard_export.sql` joins `bank_clean` to `customer_segments` into one flat table —
every column the dashboard could plausibly need, with no further joins required inside Tableau.

In [2]:
export_df = run_sql_file(con, "15_dashboard_export.sql")
print(f"{len(export_df):,} rows, {export_df.shape[1]} columns")
export_df.head(3)

45,211 rows, 20 columns


,customer_id,segment_name,age,job,marital,education,credit_in_default,balance,has_housing_loan,has_personal_loan,contact,day,month,month_order,campaign,pdays,contacted_before,previous,poutcome,subscribed
0,1,Cold: Never Contacted,58,management,married,tertiary,False,2143,True,False,unknown,5,may,5,1,-1,False,0,unknown,No
1,2,Cold: Never Contacted,44,technician,single,secondary,False,29,True,False,unknown,5,may,5,1,-1,False,0,unknown,No
2,3,Cold: Never Contacted,33,entrepreneur,married,secondary,False,2,True,True,unknown,5,may,5,1,-1,False,0,unknown,No


In [3]:
export_path = "../dashboard/bank_marketing_dashboard_data.csv"
export_df.to_csv(export_path, index=False)
print(f"Saved to {export_path}")

Saved to ../dashboard/bank_marketing_dashboard_data.csv


## 2. Confirming the export has what the dashboard needs

The dashboard needs three things: **segment sizes**, **response rates by segment**, and **at least
one filter** (month or contact type — both are in this export, so either works, or both).

In [4]:
check = export_df.groupby("segment_name").agg(
    n_customers=("customer_id", "size"),
    subscribe_rate_pct=("subscribed", lambda s: (s == "Yes").mean() * 100),
).round(2)
check["pct_of_customers"] = (check.n_customers / len(export_df) * 100).round(1)
check.sort_values("subscribe_rate_pct", ascending=False)

,n_customers,subscribe_rate_pct,pct_of_customers
segment_name,,,
Warm: Past Success,1511,64.73,3.3
Warm: Past Mixed Outcome,1840,16.68,4.1
Warm: Past No,4903,12.60,10.8
Cold: Never Contacted,36957,9.16,81.7


This matches Phase 4 exactly — as it should, since it's the same join, just flattened into one
file. The dashboard's job is to make this table's story visible at a glance, not to recompute it.

In [5]:
print("Available filter fields (either satisfies the plan's 'at least one filter' requirement):")
print(" - month:", sorted(export_df.month.unique(), key=lambda m: export_df.loc[export_df.month == m, "month_order"].iloc[0]))
print(" - contact:", sorted(export_df.contact.unique()))
print("Other fields available for further filtering or breakdown:", 
      [c for c in export_df.columns if c not in ["customer_id", "segment_name", "subscribed"]])

Available filter fields (either satisfies the plan's 'at least one filter' requirement):
 - month: ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
 - contact: ['cellular', 'telephone', 'unknown']
Other fields available for further filtering or breakdown: ['age', 'job', 'marital', 'education', 'credit_in_default', 'balance', 'has_housing_loan', 'has_personal_loan', 'contact', 'day', 'month', 'month_order', 'campaign', 'pdays', 'contacted_before', 'previous', 'poutcome']


## Phase 5 (preparation) summary

- `dashboard/bank_marketing_dashboard_data.csv` (45,211 rows, 20 columns) is the single file
  Tableau Public will load. It is not tracked in git (it's a derived export, like the CSVs in
  `data/`) and can be regenerated by rerunning this notebook.
- It carries segment membership, the response outcome, and every field useful as a filter or
  breakdown (age, job, education, balance, month, contact type, and the raw engagement-history
  columns).
- **The dashboard itself is built and published outside this notebook**, in Tableau Public
  Desktop, following `dashboard/README.md`. That file has the exact chart-by-chart, field-by-field
  steps, and this project's README will be updated with the published link once that's done.